# 04 · Water: crop water need, water use and soil moisture

**Field Shift** (NASA Space Apps 2026), data notebook 4 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.* Needs notebook 01's files in the same Drive folder.

| Output | What's in it |
|---|---|
| `data/field.json` (adds `water`) | The demo field: monthly reference evapotranspiration, MODIS water use, SMAP and ERA5-Land soil moisture since 2000 (SMAP since 2015); each season's durum water need, water use, and soil water at sowing and in spring |
| `data/cells_water.json` | For every farmland square: monthly reference evapotranspiration (for any crop's water need), durum water need, MODIS water use, and soil water at sowing and in spring, as low / typical / high, plus each season's values |

- **Reference evapotranspiration (ET₀)** is how much water a well-watered grass crop would use, worked out from ERA5-Land weather with the FAO-56 Penman-Monteith formula. A crop's **water need** is ET₀ × its crop factor (Kc) for each month, so the app can do any crop. Durum is worked out here as the example.
- **Water use** is what crops actually evaporated, from MODIS (NASA, 500 m), on rain-fed arable pixels.
- **Soil moisture** is from NASA's SMAP satellite (9 km, since 2015), with ERA5-Land and NASA POWER as checks.

**How to run (about 10 minutes):** put your project ID below, then **Runtime → Run all**.

In [ ]:
EE_PROJECT = "skilled-clover-514221-p6"    # Earth Engine Cloud project (change it if someone else runs this)
DRIVE_FOLDER = "field-shift"            # folder for this notebook's files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: you pick the input files with an upload button. True: files are shared through Google Drive
REPO = ""                               # optional, e.g. "team-owner/field-shift": used only if a file isn't in Drive

## Setup

In [ ]:
import sys, subprocess
try:
    import pyproj, shapely                  # normally already in Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj", "shapely"], check=True)

import json, math, re, time, random, hashlib, urllib.request, datetime as dt
import concurrent.futures as cf
from pathlib import Path

import ee
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import shape

if EE_PROJECT == "your-cloud-project-id":
    raise ValueError("Put your Earth Engine Cloud project ID into EE_PROJECT (first code cell), then run again.")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            f"Google Drive didn't connect ({str(e)[:80]}).\n"
            "Run this cell again. In the Google window, pick the same account and tick 'Select all' "
            "(the Google Drive box) before pressing Continue.\n"
            "If Drive can't be used at all, set USE_DRIVE = False in the first cell. The notebook will then ask you to "
            "upload notebook 01's files, and you upload its downloads to the repo yourself.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA, CACHE = BASE / "data", BASE / "cache"
for folder in (DATA, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE)

TO_LAEA = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
CELL = 1000
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
BASELINE = (2001, 2020)          # the "normal" period: at least 15 years, printed in every file

def with_retries(fn, tries=6):
    """Earth Engine sometimes answers 'Too many requests' or times out: wait, then try again."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print("  retrying after:", str(e)[:100])
            time.sleep(min(60, 2 ** k + random.random()))

CHECKS = []
def check(ok, text, warn=False, fail=None):
    """Record and print a check: ✅ passed, ⚠️ worth a look, ❌ needs fixing (fail = what to print if it didn't pass)."""
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

def num(v, nd):
    """Round a value for the JSON files; empty or missing values become null."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return int(round(v)) if nd == 0 else round(v, nd)

def lth(values, nd, min_n=15):
    """[low, typical, high] = 10th percentile, median, 90th percentile; null if fewer than min_n values."""
    v = pd.Series(values, dtype="float64").dropna()
    if len(v) < min_n:
        return [None, None, None]
    return [num(v.quantile(0.1), nd), num(v.median(), nd), num(v.quantile(0.9), nd)]

INPUTS = ["grid.geojson", "field.json"] + list(globals().get("OPTIONAL_INPUTS", []))
missing = [n for n in ["grid.geojson", "field.json"] if not (DATA / n).exists()]
if missing and IN_COLAB and not REPO:
    from google.colab import files
    print("Choose these files (downloaded from earlier notebooks) in the upload box below, all at once:")
    print("   " + ", ".join(INPUTS) + ("   (the last ones only if you have them)" if len(INPUTS) > 2 else ""))
    for fname, content in files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)        # "field (1).json" -> "field.json"
        (DATA / clean).write_bytes(content)
        print("   got", clean)

def load_optional(name):
    """A data file if it's in the working folder, else an empty dict (no questions asked)."""
    path = DATA / name
    return json.loads(path.read_text()) if path.exists() else {}

def write_field_section(part, sections, sources):
    """This notebook's part of the demo field goes in its own file, field_<part>.json; notebook 06 merges them into field.json."""
    doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                    "field_key": FIELD_KEY, "sources": sources}}
    doc.update(sections)
    return write_json(DATA / f"field_{part}.json", doc)

def load_data_file(name):
    """A data file from the working folder, or else from the repo on GitHub (if REPO is set)."""
    path = DATA / name
    if path.exists():
        return json.loads(path.read_text())
    if REPO:
        url = f"https://raw.githubusercontent.com/{REPO}/main/data/{name}"
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode()
        path.write_text(text)
        return json.loads(text)
    if IN_COLAB:
        from google.colab import files
        print(f"{name} isn't in {DATA}. Choose it in the upload box below (you can pick several of notebook 01's files at once).")
        for fname, content in files.upload().items():
            clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)       # "field (1).json" -> "field.json"
            (DATA / clean).write_bytes(content)
        if path.exists():
            return json.loads(path.read_text())
    raise FileNotFoundError(f"{name} isn't in {DATA}. Run notebook 01 first (same Google account and DRIVE_FOLDER), "
                            "or set REPO so the file can be fetched from GitHub.")

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False))
    return path.stat().st_size / 1e6

GRID = load_data_file("grid.geojson")
FIELD_DOC = load_data_file("field.json")
FIELD = FIELD_DOC["field"]
SQUARES = [f["properties"] for f in GRID["features"]]
for sq in SQUARES:            # lower-left corner in metres, read from the EEA code, e.g. 1kmE4836N1990
    e, n = re.match(r"1kmE(\d+)N(\d+)", sq["id"]).groups()
    sq["x0"], sq["y0"] = int(e) * 1000, int(n) * 1000
FARM = [sq for sq in SQUARES if sq.get("farm")]
centre = shape(FIELD["outline"]).centroid                 # the field's data are worked out at the centre of its outline
FIELD_LON, FIELD_LAT = round(centre.x, 6), round(centre.y, 6)
FIELD_KEY = hashlib.md5(json.dumps(FIELD["outline"], sort_keys=True).encode()).hexdigest()[:8]   # changes if the outline is redrawn
FIELD_PT = ee.Geometry.Point([FIELD_LON, FIELD_LAT])
FIELD_GEOM = ee.Geometry(FIELD["outline"])
FIELD_AREA_M2 = FIELD["area_ha"] * 1e4

WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
CLC = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
RAINFED = WC.eq(40).And(CLC.eq(211))          # the project's rain-fed arable mask (10 m)

def square(sq):
    return ee.Geometry.Rectangle([sq["x0"], sq["y0"], sq["x0"] + CELL, sq["y0"] + CELL], "EPSG:3035", False)

def squares_fc(batch):
    return ee.FeatureCollection([ee.Feature(square(sq), {"id": sq["id"]}) for sq in batch])

print(f"{len(SQUARES):,} grid squares ({len(FARM):,} farmland); demo field in {FIELD['square']} "
      f"({FIELD['area_ha']} ha, outline {FIELD['outline_source'].split(':')[0]})")

## 1. Settings

In [ ]:
NOTEBOOK = "notebooks/04_water.ipynb"
TODAY = dt.date.today()
LAST_SEASON = TODAY.year if TODAY.month >= 9 else TODAY.year - 1
SEASONS = list(range(2001, LAST_SEASON + 1))
START = "2000-09-01"
END = (TODAY + dt.timedelta(days=1)).isoformat()
SMAP_BASE = (2016, 2025)          # SMAP starts in April 2015, so its "normal" is the ten seasons 2016-2025

# Durum crop factor (FAO-56 Table 11/12, winter wheat, Mediterranean, November sowing): 30 + 140 + 40 + 30 days
SOW = (11, 15)
KC_STAGES = [(30, 0.70, 0.70), (140, 0.70, 1.15), (40, 1.15, 1.15), (30, 1.15, 0.30)]   # (days, Kc start, Kc end)
KC_DAILY = np.concatenate([np.linspace(a, b, n, endpoint=False) for n, a, b in KC_STAGES])

ERA5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
ERA5_M = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
SMAP = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").select(["sm_surface", "sm_rootzone"])
ET_GF = ee.ImageCollection("MODIS/061/MOD16A2GF")
ET_NRT = ee.ImageCollection("MODIS/061/MOD16A2")
PROJ = {"era5": ERA5.first().select("temperature_2m_max").projection(),
        "smap": SMAP.first().select("sm_rootzone").projection()}

SOURCES = {
    "era5_land": {"name": "ERA5-Land daily and monthly aggregates (ECMWF, Copernicus Climate Change Service)",
                  "ee_id": "ECMWF/ERA5_LAND/DAILY_AGGR, ECMWF/ERA5_LAND/MONTHLY_AGGR", "res_m": 11132,
                  "licence": "Copernicus licence (free, with attribution)",
                  "cite": "Muñoz Sabater, J. (2019). ERA5-Land hourly data from 1950 to present. C3S Climate Data Store. doi:10.24381/cds.e2161bac"},
    "modis_et": {"name": "MODIS Terra evapotranspiration 8-day 500 m (MOD16A2GF gap-filled; MOD16A2 for the latest months)",
                 "ee_id": "MODIS/061/MOD16A2GF, MODIS/061/MOD16A2", "res_m": 500, "licence": "NASA data: no restrictions on use",
                 "cite": "Running, S., Mu, Q., Zhao, M., Moreno, A. (2021). MOD16A2GF MODIS/Terra Net Evapotranspiration Gap-Filled 8-Day L4 Global 500 m SIN Grid V061. NASA LP DAAC. doi:10.5067/MODIS/MOD16A2GF.061"},
    "smap": {"name": "NASA SMAP L4 surface and root-zone soil moisture, 3-hourly 9 km (SPL4SMGP v8)", "ee_id": "NASA/SMAP/SPL4SMGP/008",
             "res_m": 9000, "licence": "Public domain (NASA Earth Science Data and Information Policy)",
             "cite": "Reichle, R. et al. (2025). SMAP L4 Global 3-hourly 9 km EASE-Grid Surface and Root Zone Soil Moisture Geophysical Data, Version 8. NASA NSIDC DAAC. doi:10.5067/T5RUATAQREF8"},
    "terraclimate": {"name": "TerraClimate monthly reference evapotranspiration (check only)", "ee_id": "IDAHO_EPSCOR/TERRACLIMATE",
                     "res_m": 4638, "licence": "Public domain (CC0)",
                     "cite": "Abatzoglou, J. T. et al. (2018). Scientific Data 5: 170191. doi:10.1038/sdata.2017.191"},
    "power": {"name": "NASA POWER daily (MERRA-2 based): rain, temperature and soil wetness, for checks at the demo field only", "ee_id": "API: power.larc.nasa.gov", "res_m": 55000,
              "licence": "NASA data: no restrictions on use", "cite": "NASA Langley Research Center POWER Project, https://power.larc.nasa.gov"},
    "fao56": {"name": "FAO-56 crop evapotranspiration guidelines (Penman-Monteith ET0, crop factors)", "ee_id": "-", "res_m": None,
              "licence": "FAO, cite the source", "cite": "Allen, R. G., Pereira, L. S., Raes, D., Smith, M. (1998). FAO Irrigation and Drainage Paper 56"},
}
print(f"Seasons {SEASONS[0]}-{SEASONS[-1]}; normal = {BASELINE[0]}-{BASELINE[1]} (SMAP: {SMAP_BASE[0]}-{SMAP_BASE[1]}); durum season {len(KC_DAILY)} days")

## 2. Which source pixel covers each square
Same method as notebook 03 (and it reuses 03's saved matches for ERA5-Land).

In [ ]:
FIELD_POINT = {"id": "field", "lon": FIELD_LON, "lat": FIELD_LAT}

def pixel_map(src, points):
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([p["lon"], p["lat"]]), {"id": p["id"]}) for p in points])
    ll = ee.Image.pixelLonLat().reproject(PROJ[src])
    fc = ll.sampleRegions(collection=pts, projection=PROJ[src], geometries=False)
    return {f["properties"]["id"]: (round(f["properties"]["longitude"], 4), round(f["properties"]["latitude"], 4))
            for f in with_retries(lambda: fc.getInfo())["features"]}

def cached_pixels(path, points, src):
    if path.exists():
        return {k: tuple(v) for k, v in json.loads(path.read_text()).items()}
    m = pixel_map(src, points)
    path.write_text(json.dumps(m))
    return m

PIX = {}
for src in PROJ:
    PIX[src] = cached_pixels(CACHE / f"03_pixels_{src}_farm.json", FARM, src)
    PIX[src].update(cached_pixels(CACHE / f"03_pixels_{src}_field_{FIELD_KEY}.json", [FIELD_POINT], src))
UNIQUE = {src: sorted(set(m.values())) for src, m in PIX.items()}
print({src: f"{len(u)} pixels" for src, u in UNIQUE.items()})

def pixel_fc(src):
    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"id": f"{lon},{lat}"}) for lon, lat in UNIQUE[src]])

def pix_key(src, point_id):
    lon, lat = PIX[src][point_id]
    return f"{lon},{lat}"

## 3. Fetch ERA5-Land weather for ET₀, ERA5-Land and SMAP soil moisture
Half a year per request for the daily weather; one request per year for SMAP. Everything is saved to Drive as it goes.

In [ ]:
E5_BANDS = {"temperature_2m_max": "tmax", "temperature_2m_min": "tmin", "dewpoint_temperature_2m": "tdew",
            "surface_net_solar_radiation_sum": "rns", "surface_net_thermal_radiation_sum": "rnl",
            "u_component_of_wind_10m": "u10", "v_component_of_wind_10m": "v10", "surface_pressure": "pres"}

def parse_rows(out, pattern):
    rows = []
    for f in with_retries(lambda: out.getInfo())["features"]:
        p = f["properties"]
        row = {"id": p["id"]}
        for k, v in p.items():
            m = re.search(pattern, k)
            if m:
                row[f"{m.group(1)}_{m.group(2)}"] = v
        rows.append(row)
    return pd.DataFrame(rows).set_index("id")

def cached_csv(path, src, fn):
    if path.exists():
        df = pd.read_csv(path, index_col="id")
        if set(df.index) >= {f"{lon},{lat}" for lon, lat in UNIQUE[src]}:
            return df
    df = fn()
    df.to_csv(path)
    return df

def no_data(src):
    return pd.DataFrame(index=pd.Index([f"{lon},{lat}" for lon, lat in UNIQUE[src]], name="id"))

def era5_daily(a, b):
    def rn(img):
        d = img.date().format("YYYYMMdd")
        return img.select(list(E5_BANDS)).rename([ee.String(n + "_d").cat(d) for n in E5_BANDS.values()])
    col = ERA5.filterDate(a, b)
    if with_retries(lambda: col.size().getInfo()) == 0:
        print(f"  no ERA5-Land data for {a} to {b} yet: skipped")
        return no_data("era5")
    out = col.map(rn).toBands().reduceRegions(collection=pixel_fc("era5"), reducer=ee.Reducer.first())
    return parse_rows(out, r"(tmax|tmin|tdew|rns|rnl|u10|v10|pres)_d(\d{8})$")

def era5_soil(a, b):
    def rn(img):
        d = img.date().format("YYYYMM")
        return img.select(["volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2", "volumetric_soil_water_layer_3"]).rename(
            [ee.String("sw1_m").cat(d), ee.String("sw2_m").cat(d), ee.String("sw3_m").cat(d)])
    col = ERA5_M.filterDate(a, b)
    if with_retries(lambda: col.size().getInfo()) == 0:
        return no_data("era5")
    out = col.map(rn).toBands().reduceRegions(collection=pixel_fc("era5"), reducer=ee.Reducer.first())
    return parse_rows(out, r"(sw1|sw2|sw3)_m(\d{6})$")

def smap_year(y):
    bands = []
    for m in range(1, 13):
        m0 = dt.date(y, m, 1)
        if m0 > TODAY or m0 < dt.date(2015, 4, 1):
            continue
        m1 = dt.date(y + (m == 12), m % 12 + 1, 1)
        c = SMAP.filterDate(m0.isoformat(), min(m1, TODAY).isoformat())
        empty = ee.Image.constant([-999, -999]).rename(["sm_surface", "sm_rootzone"])
        bands.append(ee.Image(ee.Algorithms.If(c.size().gt(0), c.mean(), empty)).rename([f"sms_m{m0:%Y%m}", f"smr_m{m0:%Y%m}"]))
    # monthly means have no projection of their own (Earth Engine would work at 1 degree), so work on the SMAP 9 km grid
    out = ee.Image.cat(bands).reduceRegions(collection=pixel_fc("smap"), reducer=ee.Reducer.first(), crs=PROJ["smap"])
    return parse_rows(out, r"(sms|smr)_m(\d{6})$")

def halves(y):
    first, second = (f"{y}-01-01", f"{y}-07-01"), (f"{y}-07-01", f"{y + 1}-01-01")
    out = [first, second] if y < TODAY.year else ([first, (f"{y}-07-01", END)] if TODAY.month >= 7 else [(f"{y}-01-01", END)])
    return [(max(a, START), b) for a, b in out if b > START]

jobs = {f"04_era5_{a}": ("era5", lambda a=a, b=b: era5_daily(a, b)) for y in range(2000, TODAY.year + 1) for a, b in halves(y)}
jobs.update({f"04_era5soil_{d}": ("era5", lambda d=d: era5_soil(f"{d}-01-01", min(f"{d + 10}-01-01", END)))
             for d in (2000, 2010, 2020)})
jobs.update({f"04_smap9km_{y}": ("smap", lambda y=y: smap_year(y)) for y in range(2015, TODAY.year + 1)})

t0, raw = time.time(), {}
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(cached_csv, CACHE / f"{name}.csv", src, fn): name for name, (src, fn) in jobs.items()}
    for i, fut in enumerate(cf.as_completed(futs), 1):
        raw[futs[fut]] = fut.result()
        if i % 10 == 0 or i == len(futs):
            print(f"  {i}/{len(futs)} requests done, {time.time() - t0:.0f} s")

def table(prefix, var, fmt):
    frames = [raw[k].filter(regex=f"^{var}_") for k in sorted(raw) if k.startswith(prefix + "_") and len(raw[k])]
    df = pd.concat(frames, axis=1)
    df.columns = pd.to_datetime([c.split("_")[1] for c in df.columns], format=fmt)
    df = df.T.sort_index()
    return df[~df.index.duplicated()]

E5 = {v: table("04_era5", v, "%Y%m%d") for v in E5_BANDS.values()}
SW = {v: table("04_era5soil", v, "%Y%m") for v in ("sw1", "sw2", "sw3")}
SM = {v: table("04_smap9km", v, "%Y%m").where(lambda d: d >= 0) for v in ("sms", "smr")}

def use_nearest_land(src, valid):
    """Squares on the coast can sit in a pixel that is sea, with no data: point them at the nearest pixel that has data."""
    good = [tuple(float(x) for x in k.split(",")) for k in valid]
    stuck = [pid for pid, (lon, lat) in PIX[src].items() if f"{lon},{lat}" not in valid]
    for pid in stuck:
        lon, lat = PIX[src][pid]
        near = min(good, key=lambda g: (g[0] - lon) ** 2 + (g[1] - lat) ** 2) if good else None
        if near and math.hypot(near[0] - lon, near[1] - lat) <= 0.15:          # at most about one pixel away
            PIX[src][pid] = near
    left = sum(1 for lon, lat in PIX[src].values() if f"{lon},{lat}" not in valid)
    return len(stuck), left

stuck, left = use_nearest_land("era5", {c for c in E5["tmax"].columns if E5["tmax"][c].notna().any()})
check(left == 0, f"ERA5-Land: {stuck} squares fall in sea pixels and use the nearest land pixel", warn=True,
      fail=f"ERA5-Land: {left} squares are in sea pixels with no land pixel nearby, so they have no ET0")
print(f"ERA5-Land: {E5['tmax'].shape[0]} days x {E5['tmax'].shape[1]} pixels; SMAP: {SM['smr'].shape[0]} months x {SM['smr'].shape[1]} pixels")

## 4. Reference evapotranspiration (FAO-56 Penman-Monteith)
Daily, from ERA5-Land's temperature, dew point, net radiation, wind and pressure. ERA5-Land's daily wind is an average of directions, which can make it a little too low, so wind at 2 m is kept at 0.5 m/s or more (FAO-56 practice). TerraClimate's ET₀ is used as a check at the field.

In [ ]:
def e0(t):
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))

def et0_pm(tmax_k, tmin_k, tdew_k, rns, rnl, u10, v10, pres):
    tmax, tmin, tdew = tmax_k - 273.15, tmin_k - 273.15, tdew_k - 273.15
    t = (tmax + tmin) / 2
    es, ea = (e0(tmax) + e0(tmin)) / 2, e0(tdew)
    delta = 4098 * e0(t) / (t + 237.3) ** 2
    gamma = 0.000665 * pres / 1000
    u2 = np.maximum(np.sqrt(u10 ** 2 + v10 ** 2) * 4.87 / np.log(67.8 * 10 - 5.42), 0.5)
    rn = (rns + rnl) / 1e6                                              # J/m2/day -> MJ/m2/day; soil heat flux ~0 for a day
    et0 = (0.408 * delta * rn + gamma * 900 / (t + 273) * u2 * (es - ea).clip(lower=0)) / (delta + gamma * (1 + 0.34 * u2))
    return et0.clip(lower=0)

ET0 = et0_pm(*(E5[v] for v in ("tmax", "tmin", "tdew", "rns", "rnl", "u10", "v10", "pres")))
print(f"ET0 worked out for {ET0.shape[1]} pixels over {ET0.shape[0]} days")

def monthly_sum(daily):
    g = daily.resample("MS")
    return g.sum(min_count=25)

def durum_need(et0_series, season):
    start = pd.Timestamp(season - 1, *SOW)
    days = et0_series[(et0_series.index >= start) & (et0_series.index < start + pd.Timedelta(days=len(KC_DAILY)))]
    if days.notna().sum() < 0.95 * len(KC_DAILY):
        return np.nan
    return float((days.to_numpy() * KC_DAILY[:len(days)]).sum())

def window_sum(daily, season, m0, m1):      # months m0..m1 inclusive, Nov-Jun style windows across the new year
    a = pd.Timestamp(season - 1 if m0 >= 9 else season, m0, 1)
    b = pd.Timestamp(season, m1, 1) + pd.offsets.MonthEnd(0)
    w = daily[(daily.index >= a) & (daily.index <= b)]
    return float(w.sum()) if w.notna().sum() >= 0.95 * ((b - a).days + 1) else np.nan

def et0_tables(pix):
    s = ET0[pix]
    seasons = {season: {"et0_season": window_sum(s, season, 11, 6), "durum_need": durum_need(s, season)} for season in SEASONS}
    months = monthly_sum(s)
    return seasons, months

def smap_tables(pix):
    smr, sms = SM["smr"][pix] * 100, SM["sms"][pix] * 100        # volume fraction -> %
    seasons = {}
    for season in SEASONS:
        nov = pd.Timestamp(season - 1, 11, 1)
        spring = [pd.Timestamp(season, 3, 1), pd.Timestamp(season, 4, 1)]
        seasons[season] = {"sm_sowing": smr.get(nov, np.nan), "sm_sowing_surface": sms.get(nov, np.nan),
                           "sm_spring": float(np.mean([smr.get(d, np.nan) for d in spring]))}
    return seasons, smr

def lth_years(tab, key, nd, years, min_n):
    return lth([v.get(key) for y, v in tab.items() if years[0] <= y <= years[1]], nd, min_n=min_n)

def month_lth(month_series, nd, years=BASELINE, min_n=15):
    m = month_series[(month_series.index.year >= years[0]) & (month_series.index.year <= years[1])]
    return [lth(m[m.index.month == k].values, nd, min_n=min_n) for k in range(1, 13)]

## 5. MODIS water use on rain-fed arable land
For every square and season: what the crops actually evaporated from November to June, from the 500 m MODIS pixels that are at least half rain-fed arable. Three requests in all.

In [ ]:
last_gf = with_retries(lambda: ET_GF.aggregate_max("system:time_start").getInfo())
gf_end = dt.datetime.fromtimestamp(last_gf / 1000, dt.timezone.utc).date() + dt.timedelta(days=8)
ET_COL = ET_GF.merge(ET_NRT.filterDate(gf_end.isoformat(), END))
EPROJ = ET_GF.first().select("ET").projection()
MASK500 = (RAINFED.unmask(0).reproject(WC.projection())
           .reduceResolution(ee.Reducer.mean(), maxPixels=4096).reproject(EPROJ).gte(0.5))
print(f"Gap-filled MODIS ET until {gf_end}; near-real-time product after that")

def et_mm(img):
    et = img.select("ET")
    return et.updateMask(et.lt(32700)).multiply(0.1)

def season_aet(season):
    c = ET_COL.filterDate(f"{season - 1}-11-01", f"{season}-07-01").map(et_mm)
    empty = ee.Image.constant(0).rename("ET").updateMask(0)              # a season with no MODIS data yet
    total = ee.Image(ee.Algorithms.If(c.size().gt(0), c.sum(), empty))
    count = ee.Image(ee.Algorithms.If(c.size().gt(0), c.count(), empty.unmask(0)))
    return total.updateMask(count.gte(26)).updateMask(MASK500).rename(f"aet_s{season}")

AET_STACK = ee.Image.cat([season_aet(s) for s in SEASONS])

def aet_batch(batch):
    # season totals have no projection of their own (Earth Engine would work at 1 degree), so work on the MODIS 500 m grid
    out = AET_STACK.reduceRegions(collection=squares_fc(batch), reducer=ee.Reducer.mean(), crs=EPROJ, tileScale=4)
    return parse_rows(out, r"(aet)_s(\d{4})$")

aet_file = CACHE / f"04_aet_cells_{SEASONS[-1]}.csv"
if aet_file.exists():
    aet = pd.read_csv(aet_file, index_col="id")
else:
    batches = [FARM[i:i + 900] for i in range(0, len(FARM), 900)]
    with cf.ThreadPoolExecutor(max_workers=3) as pool:
        aet = pd.concat(list(pool.map(aet_batch, batches)))
    aet.to_csv(aet_file)
aet.columns = [int(c.split("_")[1]) for c in aet.columns]
print(f"MODIS water use for {aet.notna().any(axis=1).sum():,} of {len(FARM):,} squares")

def field_et_rows(a, b):
    rows = with_retries(lambda: ET_COL.filterDate(a, b).select(["ET", "PET"]).getRegion(FIELD_PT, 500).getInfo())
    return pd.DataFrame(rows[1:], columns=rows[0])

fe_file = CACHE / f"04_et_field_{FIELD_KEY}.csv"
if fe_file.exists():
    fe = pd.read_csv(fe_file)
else:
    with cf.ThreadPoolExecutor(max_workers=6) as pool:
        fe = pd.concat(list(pool.map(lambda y: field_et_rows(max(f"{y}-01-01", START), min(f"{y + 1}-01-01", END)),
                                     range(2000, TODAY.year + 1))), ignore_index=True)
    fe.to_csv(fe_file, index=False)
fe["date"] = pd.to_datetime(fe["time"], unit="ms").dt.normalize().astype("datetime64[ns]")
fe["et"] = pd.to_numeric(fe["ET"], errors="coerce").where(lambda v: v < 32700) * 0.1
fe = fe.dropna(subset=["et"]).drop_duplicates("date").set_index("date").sort_index()
field_aet = {s: (lambda w: float(w.sum()) if len(w) >= 26 else np.nan)(fe["et"][(fe.index >= pd.Timestamp(s - 1, 11, 1)) & (fe.index < pd.Timestamp(s, 7, 1))])
             for s in SEASONS}
aet_month_field = fe["et"].resample("MS").sum(min_count=3)

## 6. Checks at the field: TerraClimate ET₀ and NASA POWER soil wetness

In [ ]:
tc_file = CACHE / f"04_terraclimate_{FIELD_KEY}.csv"
if tc_file.exists():
    tc = pd.read_csv(tc_file)
else:
    rows = with_retries(lambda: ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE").filterDate(START, END).select("pet")
                        .getRegion(FIELD_PT, 4638).getInfo())
    tc = pd.DataFrame(rows[1:], columns=rows[0])
    tc.to_csv(tc_file, index=False)
tc_pet = pd.Series(pd.to_numeric(tc["pet"], errors="coerce").values * 0.1,
                   index=pd.to_datetime(tc["time"], unit="ms").dt.normalize().astype("datetime64[ns]")).sort_index()
tc_pet = tc_pet[~tc_pet.index.duplicated()]

def power_soil():
    path = CACHE / f"04_power_soil_{FIELD_KEY}.csv"
    if path.exists():
        return pd.read_csv(path, index_col=0, parse_dates=True)
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point?parameters=GWETTOP,GWETROOT&community=AG"
           f"&longitude={FIELD_LON}&latitude={FIELD_LAT}&start=20150101&end={(TODAY - dt.timedelta(days=1)):%Y%m%d}&format=JSON")
    with urllib.request.urlopen(url, timeout=180) as r:
        data = json.loads(r.read().decode())
    df = pd.DataFrame(data["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df = df.replace(data.get("header", {}).get("fill_value", -999), np.nan).sort_index()
    df.to_csv(path)
    return df

try:
    pw = power_soil()
    pw_root = pw["GWETROOT"].resample("MS").mean()
except Exception as e:
    pw_root = None
    print("NASA POWER not reached, so that check is skipped:", str(e)[:100])

## 7. Put it together: every square and the demo field

In [ ]:
def anomaly(s):
    s = s.dropna()
    return s - s.groupby(s.index.month).transform("mean")

combo_key, records, cells_out = {}, {}, {}
cache_e5, cache_sm = {}, {}
for sq in FARM:
    ke, ks = pix_key("era5", sq["id"]), pix_key("smap", sq["id"])
    if ke not in cache_e5:
        cache_e5[ke] = et0_tables(ke)
    if ks not in cache_sm:
        cache_sm[ks] = smap_tables(ks)
    if (ke, ks) not in combo_key:
        (et_seasons, et_months), (sm_seasons, smr) = cache_e5[ke], cache_sm[ks]
        rec = {"et0_month": month_lth(et_months, 0),
               "et0_season": lth_years(et_seasons, "et0_season", 0, BASELINE, 15),
               "durum_need": lth_years(et_seasons, "durum_need", 0, BASELINE, 15),
               "sm_root_month": month_lth(smr.resample("MS").mean(), 1, SMAP_BASE, 8),
               "sm_sowing": lth_years(sm_seasons, "sm_sowing", 1, SMAP_BASE, 8),
               "sm_sowing_surface": lth_years(sm_seasons, "sm_sowing_surface", 1, SMAP_BASE, 8),
               "sm_spring": lth_years(sm_seasons, "sm_spring", 1, SMAP_BASE, 8),
               "durum_need_by_season": [num(et_seasons[s]["durum_need"], 0) for s in SEASONS],
               "sm_sowing_by_season": [num(sm_seasons[s]["sm_sowing"], 1) for s in SEASONS],
               "sm_spring_by_season": [num(sm_seasons[s]["sm_spring"], 1) for s in SEASONS]}
        combo_key[(ke, ks)] = f"r{len(records)}"
        records[combo_key[(ke, ks)]] = rec
    row = aet.loc[sq["id"]] if sq["id"] in aet.index else pd.Series(dtype=float)
    cells_out[sq["id"]] = {"rec": combo_key[(ke, ks)],
                           "aet_season": lth([row.get(s) for s in range(BASELINE[0], BASELINE[1] + 1)], 0),
                           "aet_by_season": [num(row.get(s), 0) for s in SEASONS]}
print(f"Water numbers for {len(cells_out):,} squares ({len(records)} distinct pixel combinations)")
n_et0 = sum(1 for c in cells_out.values() if records[c["rec"]]["durum_need"][1] is not None)
n_aet = sum(1 for c in cells_out.values() if c["aet_season"][1] is not None)
check(len(cells_out) == len(FARM), f"Water entries for all {len(FARM):,} farmland squares")
check(n_et0 >= 0.95 * len(FARM), f"ET0 and durum water need for {n_et0:,} of {len(FARM):,} squares", warn=True)
check(n_aet >= 0.9 * len(FARM), f"MODIS water use for {n_aet:,} of {len(FARM):,} squares (the others have no 500 m pixel "
      "that is mostly rain-fed arable)", warn=True)

# the demo field
fe5, fsm = pix_key("era5", "field"), pix_key("smap", "field")
(f_et, f_et_months), (f_sm, f_smr) = et0_tables(fe5), smap_tables(fsm)
sw_root = (7 * SW["sw1"][fe5] + 21 * SW["sw2"][fe5] + 72 * SW["sw3"][fe5]) / 100 * 100     # 0-100 cm, %
season_rows = [{"season": s, "et0_season": num(f_et[s]["et0_season"], 0), "durum_need": num(f_et[s]["durum_need"], 0),
                "water_use_modis": num(field_aet.get(s), 0), "sm_sowing": num(f_sm[s]["sm_sowing"], 1),
                "sm_sowing_surface": num(f_sm[s]["sm_sowing_surface"], 1), "sm_spring": num(f_sm[s]["sm_spring"], 1),
                "sm_spring_era5": num(np.mean([sw_root.get(pd.Timestamp(s, m, 1), np.nan) for m in (3, 4)]), 1)}
               for s in SEASONS]
months = pd.date_range(START, TODAY.replace(day=1).isoformat(), freq="MS")
monthly = {"months": [m.strftime("%Y-%m") for m in months],
           "et0": [num(v, 0) for v in f_et_months.reindex(months)],
           "et0_terraclimate": [num(v, 0) for v in tc_pet.reindex(months)],
           "water_use_modis": [num(v, 0) for v in aet_month_field.reindex(months)],
           "sm_root_smap": [num(v, 1) for v in f_smr.reindex(months)],
           "sm_surface_smap": [num(v, 1) for v in (SM["sms"][fsm] * 100).reindex(months)],
           "sm_root_era5": [num(v, 1) for v in sw_root.reindex(months)]}
if pw_root is not None:
    monthly["soil_wetness_power"] = [num(v, 2) for v in pw_root.reindex(months)]
typ = {"et0_season": lth_years(f_et, "et0_season", 0, BASELINE, 15), "durum_need": lth_years(f_et, "durum_need", 0, BASELINE, 15),
       "water_use_modis": lth([field_aet.get(s) for s in range(BASELINE[0], BASELINE[1] + 1)], 0),
       "sm_sowing": lth_years(f_sm, "sm_sowing", 1, SMAP_BASE, 8), "sm_spring": lth_years(f_sm, "sm_spring", 1, SMAP_BASE, 8)}
water = {
    "made_by": f"{NOTEBOOK} ({RUN_AT})", "field_key": FIELD_KEY,
    "about": "ET0 = FAO-56 Penman-Monteith reference evapotranspiration from ERA5-Land, mm. Durum need = ET0 x FAO-56 crop factor "
             "from sowing on 15 November for 240 days. Water use = MODIS actual evapotranspiration, November-June. "
             "Soil moisture = SMAP root zone (0-100 cm) as % volume: sowing = November, spring = March-April.",
    "baseline": f"{BASELINE[0]}-{BASELINE[1]}; SMAP {SMAP_BASE[0]}-{SMAP_BASE[1]}",
    "durum_kc": {"sowing": "15 November", "stages_days": [n for n, a, b in KC_STAGES], "kc": [0.70, 1.15, 0.30], "source": "fao56"},
    "calendar": {"et0_month": month_lth(f_et_months, 0), "sm_root_month": month_lth(f_smr.resample("MS").mean(), 1, SMAP_BASE, 8)},
    "typical": typ, "seasons": season_rows, "monthly": monthly,
    "sources": ["era5_land", "modis_et", "smap", "terraclimate", "power", "fao56"],
}
print("Typical at the field:", json.dumps(typ))

In [ ]:
# Checks
yearly_et0 = f_et_months[(f_et_months.index.year >= BASELINE[0]) & (f_et_months.index.year <= BASELINE[1])].groupby(lambda d: d.year).sum(min_count=12).median()
check(700 <= yearly_et0 <= 1500, f"Yearly reference evapotranspiration at the field: {yearly_et0:.0f} mm (plausible 700-1,500 for inland southern Italy)", warn=True)
both = pd.concat([f_et_months.rename("era5"), tc_pet.rename("tc")], axis=1).dropna()
ratio = both["era5"].sum() / both["tc"].sum() if len(both) else float("nan")
check(abs(ratio - 1) <= 0.2, f"ERA5-Land ET0 is {ratio:.0%} of TerraClimate's over {len(both)} common months", warn=True)
wu = typ["water_use_modis"][1]
area_wu = float(np.nanmedian([c["aet_season"][1] for c in cells_out.values() if c["aet_season"][1] is not None] or [np.nan]))
check(wu is not None and 150 <= wu <= 400, f"Water use November-June (MODIS): field {wu} mm, area median {area_wu:.0f} mm "
      "(Garofalo et al. 2009 simulated 247 mm for wheat near Foggia)", warn=True)
a_smap, a_e5 = anomaly(f_smr.resample("MS").mean()), anomaly(sw_root)
pair = pd.concat([a_smap.rename("smap"), a_e5.rename("era5")], axis=1).dropna()
r_e5 = pair.corr().iloc[0, 1] if len(pair) > 12 else float("nan")
check(r_e5 >= 0.6, f"SMAP and ERA5-Land root-zone soil moisture move together at the field: r = {r_e5:.2f} over {len(pair)} months", warn=True)
if pw_root is not None:
    pair = pd.concat([a_smap.rename("smap"), anomaly(pw_root).rename("power")], axis=1).dropna()
    r_pw = pair.corr().iloc[0, 1] if len(pair) > 12 else float("nan")
    check(r_pw >= 0.6, f"SMAP and NASA POWER soil wetness move together at the field: r = {r_pw:.2f}", warn=True)
normal_spring = typ["sm_spring"][1]
s24 = next((r["sm_spring"] for r in season_rows if r["season"] == 2024), None)
s26 = next((r["sm_spring"] for r in season_rows if r["season"] == 2026), None)
if normal_spring is not None and s24 is not None:
    check(s24 < normal_spring, f"Drought test: spring root-zone soil water 2024 = {s24}% (SMAP normal {normal_spring}%)", warn=True)
if normal_spring is not None and s26 is not None:
    check(s26 > normal_spring, f"Wet-year test: spring root-zone soil water 2026 = {s26}%", warn=True)

## 8. Write the files

In [ ]:
INDICATORS = {
    "et0_month": {"what": "reference evapotranspiration in each month, mm (multiply by a crop's Kc for its water need)", "shape": "12 months x [low, typical, high]", "source": "era5_land"},
    "et0_season": {"what": "reference evapotranspiration November-June, mm", "shape": "[low, typical, high]", "source": "era5_land"},
    "durum_need": {"what": "durum water need over its 240-day season from 15 November (ET0 x Kc), mm", "shape": "[low, typical, high]", "source": "era5_land, fao56"},
    "sm_root_month": {"what": f"root-zone soil water (0-100 cm) in each month, % volume, normal = {SMAP_BASE[0]}-{SMAP_BASE[1]}", "shape": "12 months x [low, typical, high]", "source": "smap"},
    "sm_sowing": {"what": "root-zone soil water in November (sowing), % volume", "shape": "[low, typical, high]", "source": "smap"},
    "sm_sowing_surface": {"what": "top 5 cm soil water in November, % volume", "shape": "[low, typical, high]", "source": "smap"},
    "sm_spring": {"what": "root-zone soil water in March-April, % volume", "shape": "[low, typical, high]", "source": "smap"},
    "durum_need_by_season": {"what": "durum water need of each season, mm", "shape": "one value per season in meta.seasons", "source": "era5_land, fao56"},
    "sm_sowing_by_season": {"what": "November root-zone soil water of each season (null before SMAP)", "shape": "one value per season", "source": "smap"},
    "sm_spring_by_season": {"what": "March-April root-zone soil water of each season (null before SMAP)", "shape": "one value per season", "source": "smap"},
    "aet_season": {"what": "water actually used by crops November-June (MODIS evapotranspiration on rain-fed arable pixels), mm", "shape": "[low, typical, high], in cells[id]", "source": "modis_et"},
    "aet_by_season": {"what": "MODIS water use of each season, mm", "shape": "one value per season, in cells[id]", "source": "modis_et"},
}
cells_doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                      "keys": "cells[id] holds the MODIS values (aet_*) for each farmland square and 'rec', a key into records "
                              "for the ERA5-Land and SMAP values shared by squares under the same pixels: records[cells[id].rec]",
                      "low_typical_high": f"10th percentile, median and 90th percentile of the seasons {BASELINE[0]}-{BASELINE[1]} "
                                          f"(SMAP: {SMAP_BASE[0]}-{SMAP_BASE[1]})",
                      "baseline": f"{BASELINE[0]}-{BASELINE[1]}", "seasons": SEASONS,
                      "season": "named by harvest year: season 2024 = September 2023 to August 2024",
                      "durum_kc": water["durum_kc"], "indicators": INDICATORS,
                      "sources": {k: SOURCES[k] for k in ("era5_land", "modis_et", "smap", "fao56")}},
             "cells": cells_out, "records": records}

OUTPUTS = ["field_water.json", "cells_water.json"]
check(write_field_section("water", {"water": water}, SOURCES) < 5, "field_water.json written")
mb = write_json(DATA / "cells_water.json", cells_doc)
check(mb < 5, f"cells_water.json written: {mb:.2f} MB")

## 9. One chart to explain to a farmer
Water held in the top metre of soil at the demo field, month by month, from the SMAP satellite (ERA5-Land as a check). The band is the normal range for each month.

In [ ]:
import matplotlib.pyplot as plt
s = f_smr.resample("MS").mean().dropna()
clim = s[(s.index.year >= SMAP_BASE[0]) & (s.index.year <= SMAP_BASE[1])].groupby(lambda d: d.month)
lo, hi = clim.quantile(0.1), clim.quantile(0.9)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.fill_between(s.index, [lo[d.month] for d in s.index], [hi[d.month] for d in s.index], step="mid", color="#9ecae1", alpha=0.5,
                label=f"Normal range ({SMAP_BASE[0]}-{SMAP_BASE[1]})")
ax.plot(s.index, s.values, color="#08519c", lw=2, label="SMAP root zone (0-100 cm)")
e5 = sw_root[sw_root.index >= s.index.min()]
ax.plot(e5.index, e5.values, color="#999999", lw=1, ls="--", label="ERA5-Land (check)")
for season, colour in ((2024, "#c0392b"), (2026, "#1f6fb2")):
    ax.axvspan(pd.Timestamp(season - 1, 10, 1), pd.Timestamp(season, 6, 1), color=colour, alpha=0.08)
    ax.text(pd.Timestamp(season - 1, 10, 15), ax.get_ylim()[1], f"Season {season}", va="top", fontsize=9, color=colour)
ax.set_ylabel("Soil water (% volume)")
ax.set_title("Demo field, Genzano di Lucania: water in the top metre of soil (NASA SMAP)", loc="left")
ax.legend(frameon=False, loc="lower left", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA.parent / "soil_water_field.png", dpi=150)
plt.show()

## 10. Checks, downloads and what to paste back

In [ ]:
print("\n".join(CHECKS))
bad = [c for c in CHECKS if c.startswith("❌")]
print("\nAll essential checks passed." if not bad else f"\n{len(bad)} check(s) failed: paste the lines above into the chat.")
if IN_COLAB:
    from google.colab import files
    for name in OUTPUTS:
        files.download(str(DATA / name))
    print("Downloading", ", ".join(OUTPUTS), "- upload them to the repo's data folder (replace the old ones).")

In [ ]:
print("=== Paste this back to Claude ===")
print(f"04_water run {RUN_AT}; pixels {({s: len(u) for s, u in UNIQUE.items()})}; MODIS gap-filled until {gf_end}")
print("field typical:", json.dumps(typ))
print("field monthly ET0 typical:", [m[1] for m in water["calendar"]["et0_month"]])
print("field monthly root-zone soil water typical:", [m[1] for m in water["calendar"]["sm_root_month"]])
print("field seasons 2016-:", json.dumps([r for r in season_rows if r["season"] >= 2016]))
print(f"area median water use {area_wu:.0f} mm")
print("\n".join(CHECKS))